# Notebook 01: Dataset Audit & Metadata Verification

## Research Reference
**"An approach for accurate identification and monitoring of species in mangrove forests based on multi-source spectral data and deep learning"**  
*Monterrubio-Martínez, Trujillo-Acatitla, Tuxpan-Vargas, Moreno-Casasola*  
*Ecological Informatics*, Volume 85, 2025, Article 102961. DOI: [10.1016/j.ecoinf.2024.102961](https://doi.org/10.1016/j.ecoinf.2024.102961)

## Objective
Perform a strict, non-destructive audit of the released raw data files:
1. `data/raw/Sentinel_2_satellite_reflectance_of_monospecific.xml` (EML v2.2.0 metadata)
2. `data/raw/DataBase_Sentinel_2_Mangrove_LaMancha.csv` (Surface reflectance CSV)

We investigate schema, row count, species distribution, image IDs, spectral bands, data types, null values, and systematically verify whether the released dataset corresponds to the paper's described 60,000-pixel binary dataset, 40,000-pixel four-class dataset, or another structure.

In [1]:
import os
import sys
from pathlib import Path
import xml.etree.ElementTree as ET
import json
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CSV_PATH, XML_PATH, SPECTRAL_BANDS, RESULTS_DIR

print(f"Project root: {project_root}")
print(f"CSV path exists: {CSV_PATH.exists()} ({CSV_PATH.stat().st_size / (1024*1024):.2f} MB)")
print(f"XML path exists: {XML_PATH.exists()} ({XML_PATH.stat().st_size} bytes)")

Project root: /Users/bala/Mangroves-Blue-Carbon
CSV path exists: True (191.04 MB)
XML path exists: True (16021 bytes)


## 1. XML Metadata Inspection (EML v2.2.0)

The dataset is documented via an Ecological Metadata Language (EML v2.2.0) file. Let us parse its title, abstract, spatial extent, and methodology notes.

In [2]:
tree = ET.parse(XML_PATH)
root = tree.getroot()

def get_text_by_tag(elem, tag_name):
    for child in elem.iter():
        if child.tag.endswith(tag_name):
            t = (child.text or "").strip()
            if t: return t
    return ""

title = get_text_by_tag(root, "title")
abstract_paras = [p.text.strip() for p in root.iter() if p.tag.endswith("para") and p.text]

print("=== DATASET TITLE IN XML ===")
print(title)
print("\n=== ABSTRACT & METHODOLOGY FROM XML ===")
for i, p in enumerate(abstract_paras[:3]):
    print(f"[Paragraph {i+1}]:\n{p}\n")

=== DATASET TITLE IN XML ===
Sentinel-2 satellite reflectance of monospecific mangrove forests (Rhizophora mangle, Avicennia germinans and Laguncularia racemosa) at 10 m spatial resolution in La Mancha, Veracruz, Mexico from 2015-2021

=== ABSTRACT & METHODOLOGY FROM XML ===
[Paragraph 1]:
The database contains surface reflectance values from 147 Sentinel-2 images for monospecific red (Rhizophora mangrove), black (Avicennia germinans) and white (Laguncularia racemosa) mangrove forests at the La Mancha site (19° 35' 12'' N, 96° 23' 09'' W), Veracruz, in the Gulf of Mexico, for the period 2015 to 2021. Data are labelled by species and arranged in columns including blue (band 2 - 0.490 µm), green (band 3 - 0.560 µm), red (band 4 - 0.665 µm), red edge (bands 5 - 0.705 µm, 6 - 0.740 µm, 7 - 0.783 µm and 8A - 0.865 µm), near infrared (band 8 - 0.842 µm) and shortwave infrared (bands 11 - 1.610 µm and 12 - 2.190 µm), all with a spatial resolution of 10 metres. Mangrove species distribution wa

## 2. Raw CSV Schema & Structural Audit

Now let us examine the CSV structure without loading all 200 MB blindly. We inspect the header, columns, data types, and check for missing values.

In [3]:
# Load preview first
df_preview = pd.read_csv(CSV_PATH, nrows=5)
print("CSV Columns:", list(df_preview.columns))
print("Data Types:")
print(df_preview.dtypes)
df_preview

CSV Columns: ['ID_Imagen', 'Spp', 'Band_2', 'Band_3', 'Band_4', 'Band_5', 'Band_6', 'Band_7', 'Band_8', 'Band_8A', 'Band_11', 'Band_12']
Data Types:
ID_Imagen     object
Spp           object
Band_2       float64
Band_3       float64
Band_4       float64
Band_5       float64
Band_6       float64
Band_7       float64
Band_8       float64
Band_8A      float64
Band_11      float64
Band_12      float64
dtype: object


,ID_Imagen,Spp,Band_2,Band_3,Band_4,Band_5,Band_6,Band_7,Band_8,Band_8A,Band_11,Band_12
0,L1C_T14QQG_A002597_20151221T170103,Avicennia_germinans,0.0973,0.0905,0.0679,0.0643,0.1422,0.1808,0.2279,0.0934,0.0934,0.0447
1,L1C_T14QQG_A002597_20151221T170103,Avicennia_germinans,0.1036,0.0983,0.0751,0.0665,0.1455,0.1852,0.2436,0.0964,0.0964,0.0469
2,L1C_T14QQG_A002597_20151221T170103,Avicennia_germinans,0.1016,0.0979,0.0756,0.0689,0.1478,0.1890,0.2408,0.0989,0.0989,0.0463
3,L1C_T14QQG_A002597_20151221T170103,Avicennia_germinans,0.0908,0.0820,0.0527,0.0571,0.1390,0.1802,0.2155,0.0743,0.0743,0.0310
4,L1C_T14QQG_A002597_20151221T170103,Avicennia_germinans,0.0909,0.0818,0.0529,0.0591,0.1463,0.1900,0.2247,0.0774,0.0774,0.0348


## 3. Full Dataset Streaming Audit (1.6 Million Rows)

We scan the full CSV to determine:
- Total rows
- Unique species in column `Spp`
- Total unique Sentinel-2 scenes in `ID_Imagen`
- Null values per column
- Exact value ranges (min, max) for each spectral band

In [4]:
chunk_size = 200000
total_rows = 0
spp_counts = {}
image_ids = set()
null_counts = {col: 0 for col in df_preview.columns}
min_vals = {col: float("inf") for col in SPECTRAL_BANDS}
max_vals = {col: float("-inf") for col in SPECTRAL_BANDS}

for chunk in pd.read_csv(CSV_PATH, chunksize=chunk_size):
    total_rows += len(chunk)
    
    # Species count
    vc = chunk["Spp"].value_counts().to_dict()
    for k, v in vc.items():
        spp_counts[k] = spp_counts.get(k, 0) + int(v)
        
    # Unique images
    image_ids.update(chunk["ID_Imagen"].unique())
    
    # Nulls
    for col in chunk.columns:
        null_counts[col] += int(chunk[col].isnull().sum())
        
    # Ranges
    for b in SPECTRAL_BANDS:
        b_min = float(chunk[b].min())
        b_max = float(chunk[b].max())
        if b_min < min_vals[b]: min_vals[b] = b_min
        if b_max > max_vals[b]: max_vals[b] = b_max

print(f"Total Rows: {total_rows:,}")
print(f"Unique Sentinel-2 Images: {len(image_ids)}")
print(f"Null counts across all columns: {sum(null_counts.values())}")
print("\nSpecies Distribution:")
for spp, cnt in spp_counts.items():
    print(f"  - {spp}: {cnt:,} pixels ({cnt/total_rows*100:.2f}%)")

Total Rows: 1,605,681
Unique Sentinel-2 Images: 147
Null counts across all columns: 0

Species Distribution:
  - Avicennia_germinans: 1,441,482 pixels (89.77%)
  - Rhizophora_mangle: 125,832 pixels (7.84%)
  - Laguncularia_racemosa: 38,367 pixels (2.39%)


## 4. Band Range & Value Verification

Let us examine the minimum and maximum surface reflectance recorded across the 10 bands.

In [5]:
df_ranges = pd.DataFrame({
    "Band": SPECTRAL_BANDS,
    "Min_Reflectance": [min_vals[b] for b in SPECTRAL_BANDS],
    "Max_Reflectance": [max_vals[b] for b in SPECTRAL_BANDS],
})
df_ranges

,Band,Min_Reflectance,Max_Reflectance
0,Band_2,0.0802,0.1254
1,Band_3,0.0601,0.1147
2,Band_4,0.0400,0.0936
3,Band_5,0.0369,0.6682
4,Band_6,0.0551,0.6722
5,Band_7,0.0615,0.6805
6,Band_8,0.0608,0.3681
7,Band_8A,0.0236,0.6878
8,Band_11,0.0149,0.6184
9,Band_12,0.0049,0.4603


## 5. Footprint Invariance Verification

Let us check whether every Sentinel-2 image contains the exact same number of digitized pixels.

In [6]:
pixels_per_image = total_rows / len(image_ids)
print(f"Total Pixels: {total_rows}")
print(f"Number of Scenes: {len(image_ids)}")
print(f"Exact Pixels per Scene: {pixels_per_image:.2f}")
assert total_rows % len(image_ids) == 0, "Inconsistent pixels per image"
print("Confirmed: Every scene contains exactly 10,923 digitized pixels!")

Total Pixels: 1605681
Number of Scenes: 147
Exact Pixels per Scene: 10923.00
Confirmed: Every scene contains exactly 10,923 digitized pixels!


## 6. Critical Audit Findings & Paper Discrepancy Analysis

> ### [!IMPORTANT]
> **Dataset Classification Findings:**
> 1. **Released File Identity**: The CSV `DataBase_Sentinel_2_Mangrove_LaMancha.csv` contains **1,605,681 rows** representing monospecific mangrove forest reflectance extracted from 147 Sentinel-2 scenes using QGIS polygons.
> 2. **Missing Non-Mangrove Class**: The paper states that its experiments used:
>    - A 60,000-pixel balanced binary dataset (30,000 mangrove + 30,000 non-mangrove).
>    - A 40,000-pixel balanced multiclass dataset (10,000 each of *R. mangle*, *A. germinans*, *L. racemosa*, and non-mangrove).
>    The non-mangrove class (*rainforest, crops, water, urban, dunes, and other wetlands*) is **completely absent** from the released dataset.
> 3. **Scientific Implication**: The paper's original binary and 4-class experiments **cannot be numerically reconstructed** from the provided files without fabricating data. Under strict reproduction rules, we **must not fabricate proxy data**, but instead document this as `DATA-LIMITED`, transcribe the published results for benchmark comparison, and execute data-derived experiments on the available 3 monospecific species.

In [7]:
# Export audit findings to JSON
audit_dict = {
    "file_name": "DataBase_Sentinel_2_Mangrove_LaMancha.csv",
    "file_size_mb": round(CSV_PATH.stat().st_size / (1024*1024), 2),
    "total_rows": int(total_rows),
    "total_columns": len(df_preview.columns),
    "columns": list(df_preview.columns),
    "unique_images": len(image_ids),
    "pixels_per_image": int(pixels_per_image),
    "species_distribution": {str(k): int(v) for k, v in spp_counts.items()},
    "missing_values": {str(k): int(v) for k, v in null_counts.items()},
    "band_ranges": {b: {"min": float(min_vals[b]), "max": float(max_vals[b])} for b in SPECTRAL_BANDS},
    "non_mangrove_present": False,
    "binary_60k_present": False,
    "multiclass_40k_present": False,
    "data_classification": "DATA-LIMITED (Monospecific Mangroves Only)"
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "dataset_audit.json", "w") as f:
    json.dump(audit_dict, f, indent=2)
    
print("Successfully saved audit summary to results/dataset_audit.json")

Successfully saved audit summary to results/dataset_audit.json
